In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import  re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn import metrics
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [120]:
df=pd.read_csv('P:\AI-Resume-Screener\data\dataset_cleaned.csv')

In [121]:
df.head()

,Role,Transcript,Resume,decision,Job_Description
0,E-commerce Specialist,interviewer good morning jason great meet welc...,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...
1,Game Developer,interview scene conference room table chairs a...,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...
2,Human Resources Specialist,interview setting conference room medium sized...,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...
3,E-commerce Specialist,simulated professional interview commerce spec...,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...
4,E-commerce Specialist,simulated interview interviewer good morning a...,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...


In [105]:
df.notna().sum()

Role                     10174
Resume                   10174
decision                 10174
Job_Description          10174
similarity               10174
resume_length            10174
jd_length                10174
keyword_overlap_ratio    10174
dtype: int64

In [122]:
df.isnull().sum()  

Role               0
Transcript         0
Resume             0
decision           0
Job_Description    0
dtype: int64

In [123]:
word_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000)
combined = pd.concat([
    df["Resume"],
    df["Job_Description"],
    df['Transcript']
])
word_vectorizer.fit(combined)
resume_vectorized=word_vectorizer.transform(df["Resume"])
job_description_vectorized=word_vectorizer.transform(df["Job_Description"])
transcript_vectorized=word_vectorizer.transform(df["Transcript"])

In [124]:

jd_resume_similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        resume_vectorized[i],
        job_description_vectorized[i]
    )[0][0]

    jd_resume_similarity.append(score)

In [125]:

jd_transcript_similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        transcript_vectorized[i],
        job_description_vectorized[i]
    )[0][0]

    jd_transcript_similarity.append(score)

In [126]:

resume_transcript_similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        resume_vectorized[i],
        transcript_vectorized[i]
    )[0][0]

    resume_transcript_similarity.append(score)

In [127]:
df['jd_resume_similarity'] = jd_resume_similarity
df['jd_transcript_similarity'] = jd_transcript_similarity
df['resume_transcript_similarity'] = resume_transcript_similarity
df

,Role,Transcript,Resume,decision,Job_Description,jd_resume_similarity,jd_transcript_similarity,resume_transcript_similarity
0,E-commerce Specialist,interviewer good morning jason great meet welc...,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.134046,0.057569,0.259926
1,Game Developer,interview scene conference room table chairs a...,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.109111,0.098450,0.689386
2,Human Resources Specialist,interview setting conference room medium sized...,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.162344,0.044127,0.339827
3,E-commerce Specialist,simulated professional interview commerce spec...,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.098104,0.079060,0.344337
4,E-commerce Specialist,simulated interview interviewer good morning a...,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.045244,0.067139,0.288335
...,...,...,...,...,...,...,...,...
10169,Product Manager,simulated interview product manager role candi...,sample resume diana miller diana miller contac...,reject,comprehensive job description product manager ...,0.498995,0.180097,0.322770
10170,UI Engineer,interviewer hi grace thank coming today start ...,sample resume grace taylor grace taylor contac...,reject,sample job description ui engineer job title u...,0.481589,0.091275,0.226822
10171,UI Engineer,simulated interview ui engineer role hank brow...,sample resume hank brown hank brown ui enginee...,select,job description ui engineer role job title ui ...,0.428704,0.348252,0.368487
10172,Data Engineer,simulated interview data engineer role intervi...,sample resume diana wilson diana wilson contac...,reject,comprehensive job description data engineer ro...,0.500735,0.352764,0.408274


In [128]:
df["resume_length"] = df["Resume"].apply(lambda x: len(x.split()))
df["jd_length"] = df["Job_Description"].apply(lambda x: len(x.split()))
df['Transcript_length'] = df['Transcript'].apply(lambda x: len(x.split()))

In [130]:
keyword_overlap_ratio=[]
stop_words = set(ENGLISH_STOP_WORDS)
for i in range(len(df)):
    job_words=df['Job_Description'][i].split(" ")
    resume_words=df['Resume'][i].split(" ")
    resume_words = {
        w for w in resume_words
        if w not in stop_words
    }

    jd_words = {
        w for w in job_words
        if w not in stop_words
    }
    comman=resume_words.intersection(jd_words)
    keyword_overlap_ratio.append(len(comman)/len(job_words))

df['keyword_overlap_ratio_resume_jd'] = keyword_overlap_ratio

In [131]:
keyword_overlap_ratio_transcript_jd=[]
stop_words = set(ENGLISH_STOP_WORDS)
for i in range(len(df)):
    job_words=df['Job_Description'][i].split(" ")
    transcript_words=df['Transcript'][i].split(" ")
    transcript_words = {
        w for w in transcript_words
        if w not in stop_words
    }

    jd_words = {
        w for w in job_words
        if w not in stop_words
    }
    comman=transcript_words.intersection(jd_words)
    keyword_overlap_ratio_transcript_jd.append(len(comman)/len(job_words))

df['keyword_overlap_ratio_transcript_jd'] = keyword_overlap_ratio_transcript_jd

In [132]:
df.head()

,Role,Transcript,Resume,decision,Job_Description,jd_resume_similarity,jd_transcript_similarity,resume_transcript_similarity,resume_length,jd_length,Transcript_length,keyword_overlap_ratio_resume_jd,keyword_overlap_ratio_transcript_jd
0,E-commerce Specialist,interviewer good morning jason great meet welc...,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.134046,0.057569,0.259926,253,11,318,0.272727,0.272727
1,Game Developer,interview scene conference room table chairs a...,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.109111,0.098450,0.689386,40,11,331,0.181818,0.454545
2,Human Resources Specialist,interview setting conference room medium sized...,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.162344,0.044127,0.339827,286,13,362,0.384615,0.538462
3,E-commerce Specialist,simulated professional interview commerce spec...,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.098104,0.079060,0.344337,234,11,467,0.272727,0.272727
4,E-commerce Specialist,simulated interview interviewer good morning a...,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.045244,0.067139,0.288335,271,12,323,0.250000,0.333333


In [134]:
encoder=LabelEncoder()
X=df[['resume_length','jd_length','keyword_overlap_ratio_resume_jd','jd_resume_similarity','jd_transcript_similarity','resume_transcript_similarity','Transcript_length','keyword_overlap_ratio_transcript_jd']]
y=encoder.fit_transform(df["decision"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [135]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    random_state=42
)

param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 300, 500],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

In [136]:


pred = best_model.predict(X_test)

print(accuracy_score(y_test,pred))

print(classification_report(y_test,pred))

print(confusion_matrix(y_test,pred))

0.5849983622666229
              precision    recall  f1-score   support

           0       0.60      0.54      0.57      1548
           1       0.57      0.63      0.60      1505

    accuracy                           0.58      3053
   macro avg       0.59      0.59      0.58      3053
weighted avg       0.59      0.58      0.58      3053

[[833 715]
 [552 953]]


In [138]:
print(df.groupby("decision")["keyword_overlap_ratio_transcript_jd"].describe())

           count      mean       std  min  25%  50%  75%  max
decision                                                     
reject    5114.0  0.412929  0.172141  0.0  0.3  0.4  0.5  1.0
select    5060.0  0.405655  0.145275  0.0  0.3  0.4  0.5  1.0
